# BO_test V2 — separate single-profile VAEs
This is the **whole original notebook**, with the original cell order, both profile-generator classes, experimental data preparation, BO demonstration, training loop and final plotting retained. The VAE now reconstructs one `[t, Profile]` trajectory: tensors are `(batch, 2, SEQ_LEN)` rather than `(batch, 4, SEQ_LEN)`.

In the configuration cell immediately before data generation, select `PROFILE_KIND = "T"` or `"AS"`. The following dataset cell calls only that generator and produces `(20000, 2, 8)` training and `(2000, 2, 8)` validation tensors by default. The loss, BO helpers, measured-profile example and final diagnostic plots all use that selection. Legacy `T` loss-parameter names now refer to the selected profile.

**To train both:** run the VAE section with `PROFILE_KIND = "T"`, then change it to `"AS"` and rerun from that configuration cell through the final plots (including model initialization). Each run starts a fresh model. Checkpoints are saved independently as `trajectory_vae_T.pt` and `trajectory_vae_AS.pt` in Jupyter's working directory. Changing the selector alone does not change an already trained model. Old joint-VAE weights cannot be reused.

The original CSV paths and six `.pkl` files are still required for the original experimental-data cells. For synthetic-only training, execute the imports, then start at the Fourier-feature/VAE definition cell; skip the measured-profile reconstruction cell if the experimental data were not loaded. All later validation and plotting cells work with the synthetic dataset.

Time and profile values are normalized to [0, 1]. The original measured-data example uses division by each trajectory's maximum; for deployment, use consistent physical bounds for training and inference, and restore physical time/value units after decoding. Decoder output is not a guarantee of process feasibility.

Dependencies from the original: numpy, pandas, matplotlib, scipy, scikit-learn, scikit-optimize, joblib and torch. The original `loc_8` typo is corrected to `loc_4`; `np.concat` is changed to `np.concatenate` for compatibility.


In [ ]:
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
from skopt.plots         import plot_gaussian_process
from skopt               import gp_minimize

from skopt import Optimizer
from skopt.space import Real

from scipy.interpolate import interp1d

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy


In [ ]:
import joblib 
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter


scaler_best_X       = joblib.load("scaler_best_X.pkl")
scaler_prior_to_rbf = joblib.load("scaler_prior_to_rbf.pkl")
rbf_features        = joblib.load("rbf_features.pkl")
DESC_best           = joblib.load("DESC_best.pkl")
pipe_best           = joblib.load( "pipe_best.pkl")
RBFSampler          = joblib.load( "RBFSampler.pkl")

def dx_predictor(
    input_data: pd.DataFrame,
    k_scaler_best_X: StandardScaler,
    scaler_prior_to_rbf: StandardScaler,
    rbf_features: RBFSampler,
    k_DESC_best: list[str],
    pipe_best: PLSRegression,
    )-> np.ndarray:
    
    if isinstance(input_data, pd.Series):
        input_data = input_data.to_frame().T

    X = input_data.to_numpy()

    X = scaler_prior_to_rbf.transform(X)
    X = rbf_features.transform(X)
    X = k_scaler_best_X.transform(X[:, k_DESC_best])

    return pipe_best.predict(X)

In [ ]:
noise_level = 0.1

def f(x, noise_level=noise_level):
    return -np.sin(3 * x[0]) * (1 - np.tanh(x[0] ** 1))\
           + np.random.randn() * noise_level



In [ ]:
plt.rcParams["figure.figsize"] = (8, 18)

def f_wo_noise(x):
    return f(x, noise_level=0)

In [ ]:
def BO_proc_plotter(X_min, X_max, x_init, y_init, opt, exp_title = ''):
    
    RES_ar_0 = np.array([opt.Xi[i][0] for i in range(len(opt.Xi))]).reshape(1,-1)
    RES_ar_1 = np.array([opt.yi[i] for i in range(len(opt.Xi))]).reshape(1,-1)
    
    RESS = np.concatenate((RES_ar_0,RES_ar_1),axis=0)
    
    X_grid = np.linspace(X_min, X_max, 1000).reshape(-1, 1)

    gpr = opt.models[-1]
    X_grid_transformed = opt.space.transform(X_grid.tolist())
    mean, std = gpr.predict(X_grid_transformed, return_std=True)
    ci_lower = mean - 1.96 * std
    ci_upper = mean + 1.96 * std
    
    
    plt.rcParams["figure.figsize"] = (6, 3)
    plt.rcParams.update({'font.size': 6})
    plt.scatter(x_init, y_init,marker='x',c='b', label='Preliminary expts')

    bo_label_count = 0
    for i in range(len(x_init),len(RESS[0])):
        if bo_label_count == 0:
            plt.scatter(RESS[0][i],RESS[1][i],marker='x',c='r', label='BO expts')
            bo_label_count += 1
        else:
            plt.scatter(RESS[0][i],RESS[1][i],marker='x',c='r')
        
        plt.text(RESS[0][i],RESS[1][i],f'{str(i+1)}')

    if exp_title:
        plt.title(f'{exp_title}')
        
    plt.xlabel('Queried x points')
    plt.ylabel('Function results')
    plt.plot(X_grid,[f_wo_noise([i]) for i in X_grid],'--',label='Ground truth')

    plt.plot(X_grid.ravel(),mean,'b-',label="GP Mean Prediction")
    
    plt.fill_between(X_grid.ravel(),ci_lower,ci_upper,alpha=0.2,color='blue',label="95% Confidence Interval")
    plt.grid( which='major',alpha=0.6)
    plt.minorticks_on()
    plt.grid( which='minor',alpha=0.2)
    plt.legend(fontsize=6)

In [ ]:
loc_4 = r"C:\Users\abotondo\OneDrive - purdue.edu\ECNDATA\Desktop\OMNI_0917\_OMNI_0\expttsd1.csv"
loc_1 = r"C:\Users\abotondo\OneDrive - purdue.edu\ECNDATA\Desktop\OMNI_0917\_OMNI_0 - 1\expttsd1.csv"
loc_2 = r"C:\Users\abotondo\OneDrive - purdue.edu\ECNDATA\Desktop\OMNI_0917\_OMNI_0 - 2\expttsd1.csv"
loc_3 = r"C:\Users\abotondo\OneDrive - purdue.edu\ECNDATA\Desktop\OMNI_0917\_OMNI_0 - 2\expttsd2.csv"

In [ ]:
data_1 = pd.read_csv(loc_1)
data_2 = pd.read_csv(loc_2)
data_3 = pd.read_csv(loc_3)
data_4 = pd.read_csv(loc_4)

In [ ]:
def sol_pred(X,T,wt_AS):
    R = 8.314*1e-3
    T = T + 273.15
    return np.exp(X[0] - X[1]/(R*T) - X[2]/((R*T)**2) - X[3]*np.exp(wt_AS/1e2) )
    
X = np.array([  31.58395344,  103.16394273, -111.68508174,    4.48435146])


In [ ]:
def evaluation(filepath,name='',M1=100,M2=0,if_c=False,dataplot=False, T_min = 25): 
    # ---------
    # READ DATA
    # ---------
    dat = pd.read_csv(filepath)
    names_dict = {'L50': 'L50','L10': 'L10','W90': 'W90','W50': 'W50', 'W10':'W10',   'CE10': 'CED10', 'CE50': 'CED50', 'CE90': 'CED90', 'L90': 'L90'}
    shell = {'L50': [],'L10': [],'W90': [],'W50': [], 'W10': [],   'CE10': [], 'CE50': [], 'CE90': [], 'L90': []}
    
    for i in shell.keys():
        shell[i] = dat[names_dict[i]].iloc[2:]
    shell = pd.DataFrame(shell)
    DVal2 = dx_predictor( shell,scaler_best_X,scaler_prior_to_rbf,rbf_features,DESC_best,pipe_best)

    D10 = np.array(DVal2[:,0])
    D50 = np.array(DVal2[:,1])
    D90 = np.array(DVal2[:,2])
    
    t = np.array(dat['Unnamed: 0'      ].iloc[2:].astype(float))
    T = np.array(dat['temperature_main'].iloc[2:].astype(float))
    S = np.array(dat['SeedPump'        ].iloc[2:].astype(float))
    A = np.array(dat['ASPump'          ].iloc[2:].astype(float))
    C = np.array(dat['Concentration'   ].iloc[2:].astype(float))
    C = savgol_filter(C, window_length = 300, polyorder=1, deriv=0, axis=0)
    
    valave_pos = np.array(dat['Valve_position'].iloc[2:].astype(float))
    
    # ---------
    # GET END OF EXPERIMENT
    # ---------
    #try:
    dev_from_T_min = np.array(abs(T - T_min))
    ind_end_of_exp = np.where(dev_from_T_min < 0.1)[0][-1]
   # except:
       # ind_end_of_exp = -1

    # ---------
    # GET SEEDING TIME LENGTH
    # ---------
    try:
        ind_seeding = np.where(S != 0)[0]
        l_seed = t[ind_seeding[-1]] - t[ind_seeding[0]]
    except:
        ind_seeding = -1
        l_seed = 0
        
    seed_time = np.where(S != 0)[0][0]

    # ---------
    # CALCULATE SOLUBILITY
    # ---------
    m_AS = (float(dat['Scale'].iloc[2]) - dat['Scale'].iloc[2:].astype(float))
    AS_mf = m_AS / (m_AS + 200) * 100
    sol = sol_pred(X,dat['temperature_main'].iloc[2:].astype(float),AS_mf)

    F = dat['FBRM'].iloc[2:].astype(float)

    av_t  = M1
    av_t2 = M2

    proc_length   = t[-1] - t[0]

    theor_y  = (sol.iloc[0] - sol.iloc[-1])
    actual_y = (sol.iloc[0] - C[-1])
    yiel     = 100 - abs(theor_y - actual_y) / theor_y *100
    
    D10_pr = np.mean(D10[seed_time:seed_time+av_t])
    D50_pr = np.mean(D50[seed_time:seed_time+av_t])
    D90_pr = np.mean(D90[seed_time:seed_time+av_t])

    # ---------------------
    # Generate data for BO
    # ---------------------
    
    input_vars  = {'t': t[seed_time:ind_end_of_exp] - t[seed_time], 'AS': A[seed_time:ind_end_of_exp] , 'T': T[seed_time:ind_end_of_exp], 'SEED': l_seed * 60 }
    output_vars = {'time': proc_length , 'yield': yiel, 'CSD': [D10_pr, D50_pr, D90_pr] }

    # ---------------------
    # PLOT DATA
    # ---------------------
    if dataplot:
        fig, ax = plt.subplots(1,2,figsize=(10,3))
    
        ax[1].plot(t[seed_time:ind_end_of_exp], DVal2[seed_time:ind_end_of_exp,0], '--', alpha=0.7, label=['D10'], c='r')
        ax[1].plot(t[seed_time:ind_end_of_exp], DVal2[seed_time:ind_end_of_exp,1], '--', alpha=0.7, label=['D50'], c='g')
        ax[1].plot(t[seed_time:ind_end_of_exp], DVal2[seed_time:ind_end_of_exp,2], '--', alpha=0.7, label=['D90'], c='b')
        ax[1].set_title(f'{name}: D$_1$$_0$: {np.mean(D10[ind_end_of_exp-av_t:ind_end_of_exp-av_t2]):.2f}, D$_5$$_0$: {np.mean(D50[ind_end_of_exp-av_t:ind_end_of_exp-av_t2]):.2f}, D$_9$$_0$: {np.mean(D90[ind_end_of_exp-av_t:ind_end_of_exp-av_t2]):.2f}')
        #print(f'D10: {np.mean(D10_pr):.2f}, D50: {np.mean(D50_pr):.2f}, D90: {np.mean(D90_pr):.2f}')
        #ax[1].scatter(t[seed_time:ind_end_of_exp], D10[seed_time:ind_end_of_exp],s=6)
        #ax[1].scatter(t[seed_time:ind_end_of_exp], D50[seed_time:ind_end_of_exp],s=6)
        #ax[1].scatter(t[seed_time:ind_end_of_exp], D90[seed_time:ind_end_of_exp],s=6)
    
        ax[1].scatter(t[ind_end_of_exp-av_t:ind_end_of_exp-av_t2], D10[ind_end_of_exp-av_t:ind_end_of_exp-av_t2],s=6,c='k')
        ax[1].scatter(t[ind_end_of_exp-av_t:ind_end_of_exp-av_t2], D50[ind_end_of_exp-av_t:ind_end_of_exp-av_t2],s=6,c='k')
        ax[1].scatter(t[ind_end_of_exp-av_t:ind_end_of_exp-av_t2], D90[ind_end_of_exp-av_t:ind_end_of_exp-av_t2],s=6,c='k',label='averaged values')
    
        ax[1].plot(t[seed_time:ind_end_of_exp],T[seed_time:ind_end_of_exp],label='T [C]')
        ax[1].plot(t[seed_time:ind_end_of_exp],A[seed_time:ind_end_of_exp], label='AS [g/s]',c='orange')
        ax[1].legend(loc='upper left',ncols=3,fontsize=7,facecolor='white')
        ax_fbrm = ax[1].twinx()
        ax_fbrm.plot(t[seed_time:ind_end_of_exp],F[seed_time:ind_end_of_exp],label='FBRM',alpha=0.6)
        ax_fbrm.legend(loc='upper right',fontsize=7)
        ax_fbrm.set_ylim([0,max(F[seed_time:ind_end_of_exp])*1.1])
        ax[1].set_ylim([0,120])
        
        
        ax[0].set_title(name)
        
        ax[0].plot(t[seed_time:ind_end_of_exp],T[seed_time:ind_end_of_exp],   label='T [C]', c='magenta')
        ax[0].plot(t[seed_time:ind_end_of_exp],A[seed_time:ind_end_of_exp],   label='AS [g/s]')
        ax[0].plot(t[seed_time:ind_end_of_exp],S[seed_time:ind_end_of_exp],   label='Seed [rpm]', c = 'green')
        if if_c:
            ax[0].plot(t[seed_time:ind_end_of_exp],C[seed_time:ind_end_of_exp],   label='Conc [g/L] (non smooth)',c='r')
            ax[0].plot(t[seed_time:ind_end_of_exp],sol[seed_time:ind_end_of_exp], label='sol [g/L]', c='grey')
        
        ax[0].legend(loc='upper left',ncols=3,fontsize=7,facecolor='white')
        
        ax_fbrm = ax[0].twinx()
        ax_fbrm.plot(t[seed_time:ind_end_of_exp],F[seed_time:ind_end_of_exp],label='FBRM',c='k')
        ax_fbrm.legend(loc='upper right',fontsize=7)
        ax_fbrm.set_ylim([0,max(F[seed_time:ind_end_of_exp])*1.1])
        
        ax[0].grid(True)
        ax[0].minorticks_on()
        ax[0].grid(True, which='major', linestyle='-', linewidth=0.7)
        ax[0].grid(True, which='minor', linestyle='--', linewidth=0.4, alpha=0.7)
        ax[0].tick_params(axis='both', which='both', labelsize=9)
        ax[0].set_ylim([0,180])
        
        ax[1].grid(True)
        ax[1].minorticks_on()
        ax[1].grid(True, which='major', linestyle='-', linewidth=0.7)
        ax[1].grid(True, which='minor', linestyle='--', linewidth=0.4, alpha=0.7)
        ax[1].tick_params(axis='both', which='both', labelsize=9)
        fig.tight_layout()

    return input_vars, output_vars
    

In [ ]:
inputval_1, outputval_1 = evaluation( loc_1 , name='Exp #1', M1=100, M2=70, if_c=True, dataplot=True)
inputval_2, outputval_2 = evaluation( loc_2 , name='Exp #2', M1=100, M2=70,            dataplot=True)
inputval_3, outputval_3 = evaluation( loc_3 , name='Exp #3', M1=100, M2=70,            dataplot=True)
inputval_4, outputval_4 = evaluation( loc_4 , name='Exp #4', M1=100, M2=70, if_c=True, dataplot=True)

In [ ]:
def first_x_indices_before_steps(profile, X):

    profile = np.asarray(profile)

    if profile.ndim != 1:
        raise ValueError("profile must be a 1D list or array")

    if X <= 0:
        return np.array([0, len(profile) - 1], dtype=int)

    differences = np.abs(np.diff(profile))
    
    n = min(X, len(differences))

    step_positions = np.argsort(differences)[-n:]

    step_positions = step_positions[
        np.argsort(differences[step_positions])[::-1]
    ]

    indices = np.concatenate([
        [0],
        step_positions,
        [len(profile) - 1]
    ])

    indices = np.unique(indices)

    return indices
    
    
def smooth(A):
    for i in range(1,len(A)):
        A[i-1] = (A[i-1] + A[i])/2
    return A

def denser(indices, X):
    indices = np.asarray(indices)

    # Distance between consecutive positions
    diff = np.abs(np.diff(indices))

    # Positions of the X biggest gaps
    n = min(X, len(diff))
    gap_positions = np.argsort(diff)[-n:]

    # Add one point next to biggest gap
    new_points = indices[gap_positions] + 1

    # Keep only points that are not already present
    new_points = new_points[~np.isin(new_points, indices)]

    # Add the new points to the original list
    denser_indices = np.sort(
        np.concatenate([indices, new_points])
    )

    return denser_indices

In [ ]:
def point_reducer(inputval, N_sel = 8, N_add = 4, T_prof_div = 70, AS_prof_div = 8):
    
    t_int11 = np.linspace(0, len(inputval['t'])-1,len(inputval['t']))
    
    TTT = inputval['T'][::T_prof_div]
    ASS = inputval['AS'][::AS_prof_div]
    tttT  = inputval['t'][::T_prof_div]
    tttA  = inputval['t'][::AS_prof_div]
    
    INDT = first_x_indices_before_steps( TTT, N_sel)
    INDAS = first_x_indices_before_steps( ASS, N_sel)

    
    while len(INDT) < (N_sel+N_add):
        INDT = denser(INDT,1)

        
    while len(INDAS) < (N_sel+N_add):
        INDAS = denser(INDAS,2)

    if len(INDAS) > (N_sel+N_add):
        while len(INDAS) > (N_sel+N_add):
            INDAS = np.delete(INDAS,-2)

    plt.figure(figsize=(4,2))
    plt.plot(inputval['t'], inputval['T'],alpha=0.5,c='r')
    plt.plot(inputval['t'], inputval['AS'],alpha=0.5,c='r')
    plt.plot([tttT[int(j)] for j in INDT ], [TTT[int(j)] for j in INDT ],  'o--', c='b')
    plt.plot([tttA[int(j)] for j in INDAS], [ASS[int(j)] for j in INDAS ], 'o--',c='b')
    plt.grid(True)
    plt.minorticks_on()
    plt.grid(True, which='major', linestyle='-', linewidth=0.5)
    plt.grid(True, which='minor', linestyle='--', linewidth=0.3, alpha=0.7)

    plt.show()
    print(len(INDT), len(INDAS))
    return np.array([tttT[int(j)] for j in INDT ]), np.array([tttA[int(j)] for j in INDAS ]),  np.array([TTT[int(j)] for j in INDT ]),  np.array([ASS[int(j)] for j in INDAS ])



In [ ]:
t_T, t_AS, Test_T, Test_AS = point_reducer(inputval_1, N_sel=5, N_add=3)
#t_T, t_AS, Test_T, Test_AS = point_reducer(inputval_2, N_sel=5, N_add=3)
#t_T, t_AS, Test_T, Test_AS = point_reducer(inputval_3, N_sel=5, N_add=3)
#t_T, t_AS, Test_T, Test_AS = point_reducer(inputval_4, N_sel=5, N_add=3)


In [ ]:
X_min, X_max = -4, 4
space = [Real(X_min, X_max, name='x1')]

x_init = [ [-4],[1.1],[4] ]

y_init = [f_wo_noise(i) for i in x_init]

opt = Optimizer(dimensions=space, base_estimator="GP", acq_func="EI", n_initial_points=len(x_init), random_state=42,acq_func_kwargs = {"xi": 0.00000001})

opt.tell(x_init, y_init)
BO_proc_plotter(X_min, X_max, x_init, y_init, opt, f'Initial {len(x_init)} exp.')
plt.show()    

BEST_opt_val = min(y_init)
impr = 0
curr_BO_exp = 1
n_next_exp_min = 2

Condition = True

print(f'Is number of BO expts lower than minimum BO exp? --> {curr_BO_exp<n_next_exp_min}')
print(f'Overall decision for doing a next experiment: --> {Condition}')

In [ ]:
while Condition:

    # Generate next query point
    next_x = opt.ask()

    # Evaluate next query point
    next_y = f_wo_noise(next_x)

    # Refit GPR using the New datapoint
    opt.tell(next_x, next_y)

    # Check improvement in OF
    impr = (abs(next_y) - abs(BEST_opt_val)) / abs(BEST_opt_val) * 100
    
    if next_y < BEST_opt_val:
        BEST_opt_val = next_y

    # Check conditions for stopping or continuing
    
    if curr_BO_exp <= n_next_exp_min:
        Condition = True

    else:
        Condition = impr > 20

    print(f'Step: {curr_BO_exp}')    
    print(f'Improvement in OF. value: {impr} %')    
    print(f'Is number of BO expts <= minimum BO exp? --> {curr_BO_exp<=n_next_exp_min}')
    print(f'Is improvement larger than 20 %? --> {impr > 20}')
    print(f'Overall decision for doing a next experiment: --> {Condition}')

    BO_proc_plotter(X_min, X_max, x_init, y_init, opt, f'BO. exp. no #{str(curr_BO_exp)}')
    plt.show() 
    
    curr_BO_exp += 1

In [ ]:
# ================================================================
# 1. Fourier Feature Encoding
# ================================================================
class PositionalEncoding1D(nn.Module):
    def __init__(self, num_bands=2):
        super().__init__()
        self.num_bands = num_bands

    def forward(self, x):
        # x: (batch, seq_len)
        out = [x]
        for i in range(self.num_bands):
            freq = (2.0 ** i) * math.pi
            out.append(torch.sin(freq * x))
            out.append(torch.cos(freq * x))
        return torch.cat(out, dim=-1)


# ================================================================
# 2. Residual Block
# ================================================================
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.SiLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, x):
        return x + 0.5 * self.net(x)


# ================================================================
# 3. Single-profile Trajectory VAE
# ================================================================
class ImprovedTrajectoryVAE(nn.Module):
    def __init__(
        self,
        seq_len=8,
        latent_dim=4,
        hidden_dims=(64, 32),
        num_bands=2
    ):
        super().__init__()

        self.seq_len = seq_len
        self.latent_dim = latent_dim

        h1, h2 = hidden_dims

        self.pos_enc = PositionalEncoding1D(num_bands)
        feat_dim = 1 + 2 * num_bands
        input_dim = 2 * seq_len * feat_dim

        # ---------------- ENCODER ----------------
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.LayerNorm(h1),
            nn.SiLU(),
            ResidualBlock(h1),
            nn.Linear(h1, h2),
            nn.LayerNorm(h2),
            nn.SiLU()
        )

        self.fc_mu = nn.Linear(h2, latent_dim)
        self.fc_logvar = nn.Linear(h2, latent_dim)

        # ---------------- DECODER ----------------
        # ========================================================
        # Shared decoder backbone
        # ========================================================
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, h2),
            nn.SiLU(),
            nn.Linear(h2, h1),
            nn.SiLU(),
            ResidualBlock(h1)
        )        
        # Profile profile
        self.decoder_profile = nn.Sequential(
            nn.Linear(h1, h2),
            nn.SiLU(),
            nn.Linear(h2, seq_len),
            nn.Sigmoid()
        )
        
        # Profile time coordinates
        self.decoder_t_profile = nn.Sequential(
            nn.Linear(h1, h2),
            nn.SiLU(),
            nn.Linear(h2, seq_len - 1)
        )
        


    def encode(self, x):
        # x: (batch, 2, seq_len)
        b, c, s = x.shape

        if c != 2 or s != self.seq_len:
            raise ValueError(
                f"Expected (batch, 2, {self.seq_len}), "
                f"received {tuple(x.shape)}"
            )

        x_flat = x.reshape(b * c, s)
        x_encoded = self.pos_enc(x_flat)
        h = x_encoded.reshape(b, -1)

        h = self.encoder(h)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar

    def reparameterize(self, mu, logvar):
        if not self.training:
            return mu

        logvar = torch.clamp(logvar, -12, 8)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

        
    
    def decode(self, z):
    
        h = self.decoder(z)
    
        # ========================================================
        # Profile profile
        # ========================================================
        recon_profile = self.decoder_profile(h)
    
        # Positive time intervals for profile
        raw_dt = self.decoder_t_profile(h)
        dt = F.softplus(raw_dt) + 1e-5
    
        # Normalize total duration to 1
        dt = dt / dt.sum(dim=-1, keepdim=True)
    
        zeros = torch.zeros_like(dt[:, :1])
    
        recon_t = torch.cat(
            [zeros, torch.cumsum(dt, dim=-1)],
            dim=-1
        )
    
        return torch.stack([recon_t, recon_profile], dim=1)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [ ]:
def decode_bo_suggestion(seed_amount: float, z_latent: np.ndarray) -> dict:
    model.eval()
    device = next(model.parameters()).device

    z_latent = np.asarray(z_latent, dtype=np.float32).reshape(-1)

    if z_latent.shape != (model.latent_dim,):
        raise ValueError(
            f"Expected latent shape ({model.latent_dim},), "
            f"received {z_latent.shape}"
        )

    with torch.no_grad():
        z_tensor = torch.as_tensor(
            z_latent, dtype=torch.float32, device=device
        ).unsqueeze(0)

        decoded_profile = model.decode(z_tensor)
        profile_array = decoded_profile.squeeze(0).cpu().numpy()

    return {
        "seed_amount": seed_amount,
        "executable_profile_trajectory": profile_array,
        "t": profile_array[0],
        "Profile": profile_array[1],
    }


def encode_profile_for_bo(
    seed_amount: float,
    profile_trajectory: np.ndarray,
    use_mean_only: bool = True
) -> dict:
    model.eval()
    device = next(model.parameters()).device

    profile_array = np.asarray(
        profile_trajectory, dtype=np.float32
    )

    # Accept both unbatched and single-batch inputs
    if profile_array.shape == (2, model.seq_len):
        profile_array = profile_array[np.newaxis, :, :]

    elif profile_array.shape != (1, 2, model.seq_len):
        raise ValueError(
            f"Expected (2, {model.seq_len}) or "
            f"(1, 2, {model.seq_len}), "
            f"received {profile_array.shape}"
        )

    with torch.no_grad():
        profile_tensor = torch.as_tensor(
            profile_array, dtype=torch.float32, device=device
        )

        mu, logvar = model.encode(profile_tensor)

        if use_mean_only:
            z_tensor = mu
        else:
            std = torch.exp(0.5 * torch.clamp(logvar, -12, 8))
            z_tensor = mu + torch.randn_like(std) * std

        z_latent = z_tensor.squeeze(0).cpu().numpy()

    return {
        "seed_amount": seed_amount,
        "z_latent": z_latent
    }

In [ ]:
# Generate V- and AS-profiles for VAE fitting

class ASProfDef:
    def __init__(self, t_min=0, t_max=0.2, V_min=0, V_max=1,choices=['add','add','hold','hold','add','hold'],typpe=None):
        self.V_min = V_min
        self.V_max = V_max
        self.range_of_t = np.linspace(t_min, t_max, 10)
        self.range_of_V = np.linspace(V_min, V_max, 10)
        self.choices = choices
        self.seq_len = 8
        
        self.typpe = typpe
        self.order = 'hold'
        # removed: self.ramps / self.V_0 no longer fixed here

    def reset(self):
        # NEW: draw a fresh topology and start point every reset
        if self.typpe:
            self.ramps = ['hold', 'add','hold','add','hold'] +['hold']*(self.seq_len-6)
            self.V_profile = [0]
            
        else:
            self.ramps = np.random.choice( self.choices, size=self.seq_len - 1)    
            self.V_0 = float(np.random.choice(self.range_of_V, size=1)[0])
            self.V_profile = [self.V_0]
        self.t_profile = [0]
        
    def list_conv(self,t):
        if np.shape(t):
            t = t[0]
        return t
        
    def holding(self,t_h):
        t_h = self.list_conv(t_h)
        self.t_profile.append(float(t_h))
        self.V_profile.append(float(self.V_profile[-1]))

    def resample_profile(self, t_max=17.0):
        #t_uniform = np.linspace(0, max(self.t_profile), self.seq_len)
        #V_uniform = np.interp(t_uniform, self.t_profile, self.V_profile)
        self.t_profile = self.t_profile / max(self.t_profile)
        return self.t_profile * 2 -1 , self.V_profile

    def ramp(self,t_r,V_r):
        t_r = self.list_conv(t_r)
        V_r = self.list_conv(V_r)

        self.t_profile.append(float(t_r))
        self.V_profile.append(float(V_r))
        pass

    def generate(self):
        self.reset()

        for i in self.ramps:
            # Force step duration to be at least 0.05 to avoid zero-length segments
            random_duration = np.random.uniform(0.00001, 0.5)
            
            if i == 'hold':
                self.holding(random_duration)
            elif i == 'add':
                random_v = np.random.uniform(self.V_min, self.V_max)
                self.ramp(0.01, random_v)
    
        self.t_profile = np.cumsum(self.t_profile)
        # Ensure t strictly scales between [-1, 1]
        t_max = self.t_profile[-1]
        if t_max > 0:
            self.t_profile = (self.t_profile / t_max) 
        
        return self.t_profile, self.V_profile   

    def pad_profile(self, max_len=5):
        curr_len = max(np.cumsum(self.t_profile))
        if max_len > curr_len:
            self.t_profile = np.append(self.t_profile, max_len - curr_len)
            self.V_profile = np.append(self.V_profile, self.V_profile[-1])
            return self.t_profile, self.V_profile
        else:
            print('ERROR')
            pass

In [ ]:
import torch
import torch.nn.functional as F


def free_bits_kl_loss(mu, logvar, threshold=0.05):

    logvar = logvar.clamp(-12, 8)

    kl_per_dim = -0.5 * torch.mean(
        1 + logvar - mu.pow(2) - logvar.exp(),
        dim=0
    )

    return torch.clamp(
        kl_per_dim, min=threshold
    ).sum()



import torch
import torch.nn.functional as F


def free_bits_kl_loss(mu, logvar, threshold=0.05):

    logvar = logvar.clamp(-12, 8)

    kl_per_dim = -0.5 * (
        1 + logvar - mu.pow(2) - logvar.exp()
    ).mean(dim=0)

    return torch.clamp(
        kl_per_dim, min=threshold
    ).sum()


def vae_loss_function(
    recon_x,
    target,
    mu,
    logvar,
    beta=1e-3,
    gamma_T=0.1,
    delta_T=0.0,
    alpha=0.5,
    w_time=5.0,
    w_T=1.0,
    w_dt_T=0.01,
    switch_weight=3.0,
    w_switch_T=1.0,
    return_components=False
):
    """
    VAE loss for a single (t, Profile) trajectory.
    Legacy T parameter names apply to the selected profile (T or AS).

    Channels:
        0: Selected profile time coordinates
        1: Selected profile values

    All channels assumed normalized to [0, 1].
    """

    # ========================================================
    # 1. Extract channels
    # ========================================================
    t_T_true = target[:, 0]
    T_true = target[:, 1]

    t_T_pred = recon_x[:, 0]
    T_pred = recon_x[:, 1]

    # ========================================================
    # 2. Control-point reconstruction
    # ========================================================
    loss_t_T = F.smooth_l1_loss(
        t_T_pred, t_T_true, beta=0.05
    )

    loss_T = F.smooth_l1_loss(
        T_pred, T_true, beta=0.05
    )



    recon_loss = (
        w_time * loss_t_T
        + w_T * loss_T
    )

    # ========================================================
    # 3. First differences
    # ========================================================
    dT_true = torch.diff(T_true, dim=-1)
    dT_pred = torch.diff(T_pred, dim=-1)


    # ========================================================
    # 4. Transition-aware weights
    # ========================================================
    weights_T = (
        1.0 + switch_weight * dT_true.abs()
    ).detach()


    # Weighted mean preserving per-trajectory normalization
    def weighted_mean(error, weights):
        return (
            (error * weights).sum(dim=-1)
            / weights.sum(dim=-1).clamp_min(1e-8)
        ).mean()

    # ========================================================
    # 5. Time interval reconstruction
    # ========================================================
    eps = 1e-6

    dt_T_true = torch.diff(t_T_true, dim=-1)
    dt_T_pred = torch.diff(t_T_pred, dim=-1)


    # Absolute interval errors
    abs_dt_T = weighted_mean(
        (dt_T_pred - dt_T_true).abs(),
        weights_T
    )


    # Logarithmic relative interval errors
    log_dt_T = weighted_mean(
        F.smooth_l1_loss(
            torch.log(dt_T_pred.clamp_min(eps)),
            torch.log(dt_T_true.clamp_min(eps)),
            beta=0.5,
            reduction="none"
        ),
        weights_T
    )


    # Combined absolute and relative interval penalties
    loss_dt_T = abs_dt_T + 0.1 * log_dt_T

    # ========================================================
    # 6. Trajectory-gradient reconstruction
    # ========================================================
    grad_T = weighted_mean(
        (dT_pred - dT_true).abs(),
        weights_T
    )


    # ========================================================
    # 7. Transition-time reconstruction
    # ========================================================
    # A transition occurs between two consecutive points.
    # For a short transition interval, the midpoint gives
    # an approximation of its physical occurrence time.

    t_mid_T_true = 0.5 * (
        t_T_true[:, :-1] + t_T_true[:, 1:]
    )
    t_mid_T_pred = 0.5 * (
        t_T_pred[:, :-1] + t_T_pred[:, 1:]
    )


    # Weight only transitions, not constant segments
    event_T = dT_true.abs().detach()

    loss_switch_T = weighted_mean(
        (t_mid_T_pred - t_mid_T_true).abs(),
        event_T
    )


    # ========================================================
    # 8. Optional temperature second differences
    # ========================================================
    if delta_T > 0:
        curv_T_true = torch.diff(
            T_true, n=2, dim=-1
        )

        curv_T_pred = torch.diff(
            T_pred, n=2, dim=-1
        )

        curv_T = F.l1_loss(
            curv_T_pred, curv_T_true
        )
    else:
        curv_T = recon_x.new_zeros(())

    # ========================================================
    # 9. Endpoint reconstruction
    # ========================================================
    endpoint_T = (
        F.mse_loss(T_pred[:, 0], T_true[:, 0])
        + F.mse_loss(T_pred[:, -1], T_true[:, -1])
    )


    endpoint_loss = alpha * (
        w_T * endpoint_T
    )

    # ========================================================
    # 10. KL divergence
    # ========================================================
    kl_loss = free_bits_kl_loss(
        mu, logvar, threshold=0.05
    )

    # ========================================================
    # 11. Total loss
    # ========================================================
    components = {
        "reconstruction": recon_loss,
        "dt_T": w_dt_T * loss_dt_T,
        "grad_T": gamma_T * grad_T,
        "switch_T": w_switch_T * loss_switch_T,
        "curvature_T": delta_T * curv_T,
        "endpoints": endpoint_loss,
        "KL": beta * kl_loss
    }

    total_loss = sum(components.values())

    if return_components:
        return total_loss, {
            k: v.detach().item()
            for k, v in components.items()
        }

    return total_loss

In [ ]:
# Generate V- and AS-profiles for VAE fitting

class ProfDef:
    def __init__(self, t_min=0, t_max=0.2, V_min=-1, V_max=1,choices=['ramp','hold','hold','hold','ramp','hold'],typpe=None):
        self.V_min = V_min
        self.V_max = V_max
        self.range_of_t = np.linspace(t_min, t_max, 10)
        self.range_of_V = np.linspace(V_min, V_max, 10)
        self.choices = choices
        self.seq_len = 8
        self.typpe = typpe
        # removed: self.ramps / self.V_0 no longer fixed here

    def reset(self):
        
        if self.typpe:
            self.V_profile = [1.0]
            
        else:
            self.V_0 = float(np.random.choice(self.range_of_V, size=1)[0])
            self.V_profile = [self.V_0]
            
        self.ramps = np.random.choice( self.choices, size=self.seq_len - 1)    
             
        self.t_profile = [0]
        
    def list_conv(self,t):
        if np.shape(t):
            t = t[0]
        return t
        
    def holding(self,t_h):
        t_h = self.list_conv(t_h)
        self.t_profile.append(float(t_h))
        self.V_profile.append(float(self.V_profile[-1]))

    def resample_profile(self, t_max=17.0):
        #t_uniform = np.linspace(0, max(self.t_profile), self.seq_len)
        #V_uniform = np.interp(t_uniform, self.t_profile, self.V_profile)
        self.t_profile = self.t_profile / max(self.t_profile)
        return self.t_profile * 2 -1 , self.V_profile

    def ramp(self,t_r,V_r):
        t_r = self.list_conv(t_r)
        V_r = self.list_conv(V_r)
        if self.typpe:
            if t_r > 0.1:
                t_r = 0.1
        self.t_profile.append(float(t_r))
        self.V_profile.append(float(V_r))
        pass

    def generate(self):
        self.reset()

        for i in self.ramps:
            # Force step duration to be at least 0.05 to avoid zero-length segments
            random_duration = np.random.uniform(0.05, 1.0)
            
            if i == 'hold':
                self.holding(random_duration)
            elif i == 'ramp':
                random_v = np.random.uniform(self.V_min, min(self.V_profile))
                self.ramp(random_duration, random_v)
    
        self.t_profile = np.cumsum(self.t_profile)
        # Ensure t strictly scales between [-1, 1]
        t_max = self.t_profile[-1]
        if t_max > 0:
            self.t_profile = (self.t_profile / t_max) 
        
        return self.t_profile, self.V_profile   

    def pad_profile(self, max_len=5):
        curr_len = max(np.cumsum(self.t_profile))
        if max_len > curr_len:
            self.t_profile = np.append(self.t_profile, max_len - curr_len)
            self.V_profile = np.append(self.V_profile, self.V_profile[-1])
            return self.t_profile, self.V_profile
        else:
            print('ERROR')
            pass

In [ ]:
SEQ_LEN = 8
LATENT_DIM = 7
BATCH_SIZE = 64

# Train this notebook once for T, then once for AS. Each run saves a separate model.
PROFILE_KIND = "T"  # Change to "AS" for the antisolvent VAE.
assert PROFILE_KIND in ("T", "AS")

In [ ]:
dataset_list_VAL = []
dataset_list_PCA = []

N_sampl     = 20000
N_sampl_VAL = 2000

T_gen = ProfDef(V_min=0.0, V_max=1.0,t_min=0.2, t_max=1.0,choices=['ramp','hold','hold','hold','hold','hold'],typpe='T')
AS_gen = ASProfDef(V_min=0.0, V_max=1.0,t_min=0, t_max=1.0,choices=['add','add','hold','hold','hold'],typpe='AS')

# Only the selected generator is called. Both original generator classes are unchanged.
profile_gen = T_gen if PROFILE_KIND == "T" else AS_gen
profile_gen.seq_len = SEQ_LEN
if PROFILE_KIND == "AS" and SEQ_LEN < 6:
    raise ValueError("The original AS generator requires SEQ_LEN >= 6")

dataset_list = []

while len(dataset_list) < N_sampl:
    t, profile = profile_gen.generate()
    if np.ptp(profile) < 1e-6:
        continue
    if len(profile) == SEQ_LEN:
        dataset_list.append([t, profile])
synthetic_dataset = torch.tensor(np.array(dataset_list), dtype=torch.float32)

while len(dataset_list_VAL) < N_sampl_VAL:
    t, profile = profile_gen.generate()
    if np.ptp(profile) < 1e-6:
        continue
    if len(profile) == SEQ_LEN:
        dataset_list_VAL.append([t, profile])
synthetic_dataset_VAL = torch.tensor(np.array(dataset_list_VAL), dtype=torch.float32)

In [ ]:
print(np.shape(synthetic_dataset))
print(np.shape(synthetic_dataset_VAL))


In [ ]:
N = np.random.randint(0, len(synthetic_dataset))
print(N)
plt.plot(synthetic_dataset[N][0], synthetic_dataset[N][1], 'o-')
plt.xlabel('Normalized time')
plt.ylabel(PROFILE_KIND)
plt.show()

In [ ]:
# ============================================================
# Initialize model and optimizer
# ============================================================
model = ImprovedTrajectoryVAE(
    seq_len=SEQ_LEN,
    latent_dim=LATENT_DIM,
    hidden_dims=[64, 32],
    num_bands=2
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau( optimizer, mode='min', factor=0.5, patience=30)

EPOCHS = 300
BETA_MAX = 1e-3
KL_WARMUP = 50

best_val_loss = float('inf')
best_model_state = None

train_losses = []
val_losses = []

In [ ]:
# ============================================================
# Initialization
# ============================================================
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=15
)

EPOCHS = 300
BETA_MAX = 1e-3
KL_WARMUP = 30

best_val_loss = float('inf')
best_model_state = None
best_epoch = None

train_losses = []
val_losses = []

# Shared loss configuration
loss_params = dict(
    gamma_T=0.1 if PROFILE_KIND == "T" else 0.5,
    delta_T=0.1 if PROFILE_KIND == "T" else 0.0,
    alpha=2,
    w_time=1.0,
    w_T=1.5 if PROFILE_KIND == "T" else 1.0,
    w_dt_T=0.01 if PROFILE_KIND == "T" else 0.02,
    w_switch_T=1.0 if PROFILE_KIND == "T" else 2.0,
    switch_weight=3.0
)

# Use the same device as the model
device = next(model.parameters()).device

# Move datasets to the model's device
synthetic_dataset = synthetic_dataset.to(device)
synthetic_dataset_VAL = synthetic_dataset_VAL.to(device)

# ============================================================
# Training
# ============================================================
for epoch in range(EPOCHS):

    # KL warmup
    beta = BETA_MAX * min(
        1.0, (epoch + 1) / KL_WARMUP
    )

    model.train()

    permutation = torch.randperm(
        synthetic_dataset.size(0),
        device=device
    )

    epoch_loss = 0.0

    for i in range(
        0, synthetic_dataset.size(0), BATCH_SIZE
    ):

        indices = permutation[i:i + BATCH_SIZE]
        batch_x = synthetic_dataset[indices]

        optimizer.zero_grad()

        recon_x, mu, logvar = model(batch_x)

        # Correct training target and KL weight
        loss = vae_loss_function(
            recon_x,
            batch_x,
            mu,
            logvar,
            beta=beta,
            **loss_params
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        epoch_loss += loss.item() * batch_x.size(0)

    epoch_loss /= synthetic_dataset.size(0)

    # ========================================================
    # Validation
    # ========================================================
    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for i in range(
            0, synthetic_dataset_VAL.size(0), BATCH_SIZE
        ):

            batch_val = synthetic_dataset_VAL[
                i:i + BATCH_SIZE
            ]

            recon_val, mu_val, logvar_val = model(
                batch_val
            )

            # Correct validation reconstruction and target
            v_loss = vae_loss_function(
                recon_val,
                batch_val,
                mu_val,
                logvar_val,
                beta=beta,
                **loss_params
            )

            val_loss += v_loss.item() * batch_val.size(0)

    val_loss /= synthetic_dataset_VAL.size(0)

    # ========================================================
    # Store loss history
    # ========================================================
    train_losses.append(epoch_loss)
    val_losses.append(val_loss)

    # ========================================================
    # Scheduler and best checkpoint
    # ========================================================
    # Compare models after KL warmup, when beta is fixed
    if epoch + 1 >= min(KL_WARMUP, EPOCHS):

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_model_state = copy.deepcopy(
                model.state_dict()
            )

    # ========================================================
    # Logging
    # ========================================================
    if (epoch + 1) % 10 == 0 or epoch == 0:

        print(
            f"Epoch {epoch + 1}/{EPOCHS} "
            f"- Train Loss: {epoch_loss:.8f} "
            f"- Val Loss: {val_loss:.8f} "
            f"- Beta: {beta:.6f} "
            f"- LR: {optimizer.param_groups[0]['lr']:.6e}"
        )

# ============================================================
# Restore best model
# ============================================================
if best_model_state is not None:
    model.load_state_dict(best_model_state)

model.eval()

print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.8f}")


# Distinct files let both independently trained VAEs coexist.
checkpoint_path = f"trajectory_vae_{PROFILE_KIND}.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "profile_kind": PROFILE_KIND,
    "seq_len": SEQ_LEN,
    "latent_dim": LATENT_DIM,
    "hidden_dims": [64, 32],
    "num_bands": 2,
    "best_epoch": best_epoch,
    "best_val_loss": best_val_loss,
}, checkpoint_path)
print(f"Saved {PROFILE_KIND} VAE to {checkpoint_path}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))

plt.plot(train_losses, label='Training loss')
plt.plot(val_losses, label='Validation loss')

plt.axvline(
    KL_WARMUP - 1,
    color='gray',
    linestyle='--',
    label='End of KL warmup'
)

plt.xlabel('Epoch')
plt.ylabel('VAE loss')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
t_T, t_AS, Test_T, Test_AS = point_reducer(inputval_1, N_sel=5, N_add=3)
#Test_T[0] = Test_T[2]
#Test_T[1] = Test_T[2]


# Select just one of the two measured trajectories.
selected_t, selected_profile = (t_T, Test_T) if PROFILE_KIND == "T" else (t_AS, Test_AS)
if max(selected_t) <= 0 or max(selected_profile) <= 0:
    raise ValueError("Expected positive time duration and profile scaling factor")
test_profile = np.array([selected_t / max(selected_t),
                         selected_profile / max(selected_profile)], dtype=np.float32).reshape(1, 2, SEQ_LEN)

def little_plotter(test_profile):

    bo_initial_point = encode_profile_for_bo(1, test_profile    )

    next_experiment = decode_bo_suggestion(1, bo_initial_point["z_latent"]    )

    reconstructed = next_experiment["executable_profile_trajectory"    ]

    fig, ax = plt.subplots(1, 1, figsize=(5, 2))
    ax.plot(test_profile[0, 0], test_profile[0, 1], 'o-', label=f'Actual {PROFILE_KIND}')
    ax.plot(reconstructed[0], reconstructed[1], 'o--', label=f'Reconstructed {PROFILE_KIND}')
    ax.legend()
    ax.set_title(PROFILE_KIND)
    ax.set_xlabel('Normalized time')
    plt.tight_layout()
    plt.show()

    return bo_initial_point, next_experiment


bo_initial_point, next_experiment = little_plotter(test_profile)

In [ ]:
def diff(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    return np.mean(np.abs(A - B))


def random_plott(synthetic_dataset_np, N_sampl, num_col=5):
    if isinstance(synthetic_dataset_np, torch.Tensor):
        synthetic_dataset_np = synthetic_dataset_np.detach().cpu().numpy()
    fig, ax = plt.subplots(1, num_col, figsize=(3 * num_col, 2.5), squeeze=False)
    N_sampl = min(N_sampl, len(synthetic_dataset_np))
    for i in range(num_col):
        NUM = np.random.randint(0, N_sampl)
        test_profile = np.asarray(synthetic_dataset_np[NUM]).reshape(1, 2, model.seq_len)
        bo_initial_point = encode_profile_for_bo(1, test_profile)
        next_experiment = decode_bo_suggestion(1, bo_initial_point['z_latent'])
        recon = next_experiment['executable_profile_trajectory']
        diff_val = diff(test_profile[0], recon)
        colorr = 'red' if diff_val > 0.05 else 'green'
        ax[0, i].plot(test_profile[0, 0], test_profile[0, 1], label=f'Original {PROFILE_KIND}', c='k')
        ax[0, i].plot(recon[0], recon[1], '--', label='VAE reconstruction', c=colorr)
        ax[0, i].set_title(f'{diff_val:.4f}', c=colorr)
        ax[0, i].set_ylim([-0.1, 1.1])
        ax[0, i].grid(True)
        ax[0, i].minorticks_on()
        ax[0, i].grid(True, which='minor', linestyle='--', linewidth=0.2, alpha=0.7)
        ax[0, i].set_ylabel(PROFILE_KIND)
        ax[0, i].set_xlabel('Normalized time')
    ax[0, 0].legend()
    fig.tight_layout()
    plt.show()


def general_plot(synthetic_dataset_VAL, NUM=200):
    if isinstance(synthetic_dataset_VAL, torch.Tensor):
        data = synthetic_dataset_VAL.detach().cpu().numpy()
    else:
        data = np.asarray(synthetic_dataset_VAL)
    for i in range(len(data)):
        test_profile = data[i].reshape(1, 2, model.seq_len)
        bo_initial_point = encode_profile_for_bo(1, test_profile)
        next_experiment = decode_bo_suggestion(1, bo_initial_point['z_latent'])
        recon = next_experiment['executable_profile_trajectory']
        diff_val = diff(test_profile[0], recon)
        colorr = 'red' if diff_val > 0.05 else 'green'
        if i % NUM == 0:
            if i > 0:
                plt.tight_layout()
                plt.show()
            fig, ax = plt.subplots(1, 1, figsize=(14, 3))
        ii = i % NUM
        ax.plot(test_profile[0, 0] + ii, test_profile[0, 1], c='k')
        ax.plot(recon[0] + ii, recon[1], '--', c=colorr)
        ax.set_ylim([-0.1, 1.1])
        ax.grid(True)
        ax.minorticks_on()
        ax.grid(True, which='minor', linestyle='--', linewidth=0.2, alpha=0.7)
        ax.set_ylabel(PROFILE_KIND)
        ax.set_xlabel('Normalized time + profile index')
    if len(data) > 0:
        plt.tight_layout()
        plt.show()

In [ ]:
NUM_great_diff = []
Great_diff = []
pred_t = []
actual_t = []
pred_profile = []
actual_profile = []

if isinstance(synthetic_dataset_VAL, torch.Tensor):
    val_data = synthetic_dataset_VAL.detach().cpu().numpy()
else:
    val_data = np.asarray(synthetic_dataset_VAL)

for i in range(len(val_data)):
    test_profile = val_data[i].reshape(1, 2, model.seq_len)
    bo_initial_point = encode_profile_for_bo(1, test_profile)
    next_experiment = decode_bo_suggestion(1, bo_initial_point['z_latent'])
    recon = next_experiment['executable_profile_trajectory']
    DD = diff(test_profile[0], recon)
    if DD > 0.05:
        NUM_great_diff.append(i)
        Great_diff.append(float(DD))
    pred_t.append(recon[0])
    actual_t.append(test_profile[0, 0])
    pred_profile.append(recon[1])
    actual_profile.append(test_profile[0, 1])

print(f'Number of poorly reconstructed profiles: '
      f'{len(NUM_great_diff)/len(val_data)*100:.2f}%, num: {len(NUM_great_diff)}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 4))

data_pairs = [
    (actual_t, pred_t, f'{PROFILE_KIND} time'),
    (actual_profile, pred_profile, f'{PROFILE_KIND} values')
]

for axis, (actual, pred, title) in zip(ax.flat, data_pairs):

    actual = np.asarray(actual).flatten()
    pred = np.asarray(pred).flatten()
    mae = np.mean(np.abs(actual - pred))
    axis.scatter(actual, pred, s=5, alpha=0.05 )
    axis.plot( [-0, 1], [-0, 1], 'k--', linewidth=1)
    axis.plot( [-0., 1.0], [0.1, 1.1], 'g--', linewidth=1)
    axis.plot( [-0., 1.0], [-0.1, 0.9], 'g--', linewidth=1)

    axis.set_xlim([-0.05, 1.05])
    axis.set_ylim([-0.05, 1.05])
    axis.set_aspect('equal', adjustable='box')

    axis.set_xlabel('Actual')
    axis.set_ylabel('Reconstructed')
    axis.set_title(f'{title} | MAE = {mae:.4f}')
    axis.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
random_plott(synthetic_dataset_VAL, 2000, num_col=5)
random_plott(synthetic_dataset_VAL, 2000, num_col=5)
random_plott(synthetic_dataset_VAL, 2000, num_col=5)
random_plott(synthetic_dataset_VAL, 2000, num_col=5)

In [ ]:
general_plot(synthetic_dataset_VAL[0:-1:5],NUM=10)